In [1]:
# Cell 1: Data Loading and Preprocessing
# Purpose: Load dataset, normalize the 'Amount' feature, and define a target distribution (16 bins)

import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# Load dataset
data = pd.read_csv('Ghita_creditcard.csv')

# Sample to ensure balanced dataset (950 normal, 50 fraud)
normal_data = data[data['Class'] == 0].sample(950, random_state=42).reset_index(drop=True)
fraud_data = data[data['Class'] == 1].sample(50, random_state=42).reset_index(drop=True)
data = pd.concat([normal_data, fraud_data], ignore_index=True)

# Preprocess Amount
amounts = data['Amount'].values
amounts_log = np.log1p(amounts)  # Log-transform to reduce skew
scaler = MinMaxScaler()
amounts_normalized = scaler.fit_transform(amounts_log.reshape(-1, 1)).flatten()
labels = data['Class'].values

# Define target distribution using only normal transactions
bins_amount = np.linspace(0, 1, 17)  # 16 bins
amount_bins = np.digitize(amounts_normalized, bins_amount) - 1
amount_bins = np.clip(amount_bins, 0, 15)
normal_indices = np.where(labels == 0)[0]

# Histogram of bin counts (normalized)
target_dist = np.histogram(amount_bins[normal_indices], bins=16, range=(0, 16), density=True)[0]
target_dist = target_dist / np.sum(target_dist)  # Normalize

print(" Cell 1 completed: Data loaded, normalized, and target distribution defined.")


 Cell 1 completed: Data loaded, normalized, and target distribution defined.


In [2]:
# Cell 2: QCBM Circuit Definition (1D)
# Purpose: Define an 8-qubit, 8-layer QCBM circuit and correctly generate Figure 1

from qiskit.visualization import circuit_drawer
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter

# Define parameters
num_qubits = 8
num_layers = 8
num_params = num_qubits * num_layers
theta = [Parameter(f'theta_{i}') for i in range(num_params)]

# Build the QCBM circuit
qc = QuantumCircuit(num_qubits)
param_idx = 0

for layer in range(num_layers):
    for qubit in range(num_qubits):
        qc.ry(theta[param_idx], qubit)
        param_idx += 1

    # Circular entanglement
    for qubit in range(num_qubits):
        qc.cx(qubit, (qubit + 1) % num_qubits)

# Measurement (optional here since we’re visualizing structure)
qc.measure_all()

# Save the circuit diagram using circuit_drawer directly
circuit_drawer(qc, output='mpl', filename='Figure_1_QCBM_Circuit.png')

print(" Cell 2 updated: QCBM circuit properly visualized and Figure 1 saved.")


 Cell 2 updated: QCBM circuit properly visualized and Figure 1 saved.


In [3]:
# Cell 3: Training Setup and Cost Function (1D)
# Purpose: Define the simulator and cost function (JSD + KL) for QCBM optimization

from qiskit_aer import AerSimulator
from scipy.stats import entropy

# Initialize Qiskit simulator
simulator = AerSimulator()

# Define Jensen-Shannon Divergence (JSD)
def jsd(p, q):
    p = np.clip(p, 1e-10, 1)
    q = np.clip(q, 1e-10, 1)
    p = p / np.sum(p)
    q = q / np.sum(q)
    m = 0.5 * (p + q)
    return 0.5 * (entropy(p, m, base=2) + entropy(q, m, base=2))

# Define Kullback-Leibler Divergence (KL)
def kl_div(p, q):
    p = np.clip(p, 1e-10, 1)
    q = np.clip(q, 1e-10, 1)
    p = p / np.sum(p)
    q = q / np.sum(q)
    return entropy(p, q, base=2)

# Track cost values for visualization
cost_values = []

# Cost function for optimization
def cost_function(params):
    # Assign parameters to circuit
    param_dict = {theta[i]: params[i] for i in range(len(params))}
    qc_bound = qc.assign_parameters(param_dict)

    # Run simulation
    result = simulator.run(qc_bound, shots=50000).result()
    counts = result.get_counts()

    # Full state probabilities (256 bins for 8 qubits)
    qcbm_probs_full = np.array([counts.get(f"{i:08b}", 0) / 50000 for i in range(256)])
    qcbm_probs_full += 1e-9
    qcbm_probs_full /= np.sum(qcbm_probs_full)

    # Coarse-grain to 16 bins
    qcbm_probs_coarse = np.array([
        np.sum(qcbm_probs_full[i*16:(i+1)*16])
        for i in range(16)
    ])
    qcbm_probs_coarse /= np.sum(qcbm_probs_coarse)

    # Compute JSD + KL cost
    jsd_value = jsd(qcbm_probs_coarse, target_dist)
    kl_value = kl_div(qcbm_probs_coarse, target_dist)
    cost = jsd_value + kl_value

    # Log cost value
    cost_values.append(cost)

    return cost

print(" Cell 3 completed: Training simulator and cost function defined.")


 Cell 3 completed: Training simulator and cost function defined.


In [4]:
# Cell 4: Train QCBM (1D) using scipy.optimize without printing progress
# Purpose: Optimize QCBM parameters using COBYLA and generate Figure 2

import numpy as np
from scipy.optimize import minimize
import matplotlib.pyplot as plt

# Generate random initial parameters
initial_params = np.random.uniform(0, 2 * np.pi, len(theta))

# Clear previous cost values
cost_values.clear()

# Run optimization with COBYLA, no printed output
result = minimize(
    cost_function,
    x0=initial_params,
    method='COBYLA',
    options={'maxiter': 250, 'disp': False}
)

# Assign optimal parameters to the circuit
param_dict = {theta[i]: result.x[i] for i in range(len(result.x))}
qc_optimized = qc.assign_parameters(param_dict)

# Simulate final circuit
result_sim = simulator.run(qc_optimized, shots=50000).result()
counts = result_sim.get_counts()

# Compute coarse-grained distribution
qcbm_probs = np.array([counts.get(f"{i:08b}", 0) / 50000 for i in range(256)])
qcbm_probs += 1e-9
qcbm_probs /= np.sum(qcbm_probs)
qcbm_probs_coarse = np.array([
    np.sum(qcbm_probs[i*16:(i+1)*16])
    for i in range(16)
])
qcbm_probs_coarse /= np.sum(qcbm_probs_coarse)

# Compute final divergence scores
jsd_final = jsd(qcbm_probs_coarse, target_dist)
kl_final = kl_div(qcbm_probs_coarse, target_dist)

print(" Cell 4 completed: QCBM training (COBYLA) finished.")
print(f"Final JSD: {jsd_final:.4f}, Final KL: {kl_final:.4f}, Total Cost: {jsd_final + kl_final:.4f}")

# Plot cost over iterations (Figure 2)
plt.figure(figsize=(12, 6))
plt.plot(range(len(cost_values)), cost_values, color='blue', label='Cost (JSD + KL)')
plt.xlabel('Evaluation')
plt.ylabel('Cost')
plt.title('Figure 2: Cost Function Over Evaluations During QCBM Training (COBYLA)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('Figure_2_Cost_Function.png')
plt.close()


 Cell 4 completed: QCBM training (COBYLA) finished.
Final JSD: 0.0031, Final KL: 0.0125, Total Cost: 0.0156


In [5]:
# Cell 5: Visualize Distributions
# Purpose: Compare target vs. QCBM (1D) distribution using side-by-side bar chart (Figure 3)

import matplotlib.pyplot as plt
import numpy as np

# Bar positions and width
bar_width = 0.4
x = np.arange(len(target_dist))

# Create the plot
plt.figure(figsize=(12, 6))
plt.bar(x - bar_width/2, target_dist, bar_width, label='Target Distribution (Normal)', color='blue', alpha=0.7)
plt.bar(x + bar_width/2, qcbm_probs_coarse, bar_width, label='QCBM Distribution', color='green', alpha=0.7)

# Labels and styling
plt.xlabel('Bin Index')
plt.ylabel('Probability Density')
plt.title('Figure 3: Distribution Comparison — Target vs. QCBM')
plt.xticks(x, range(len(target_dist)))
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Save the figure
plt.savefig('Figure_3_Distribution_Comparison.png')
plt.close()

print(" Cell 5 completed: Figure 3 generated — Target vs. QCBM distribution.")


 Cell 5 completed: Figure 3 generated — Target vs. QCBM distribution.


In [6]:
# Cell 6: Anomaly Detection with QCBM (1D)
# Purpose: Evaluate likelihoods using QCBM, detect anomalies, and visualize distributions (Figure 4)

from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

# Function to evaluate anomaly detection with QCBM
def evaluate_anomaly_detection(qcbm_probs, threshold_percentile, amounts, labels):
    bins_amount = np.linspace(0, 1, 17)  # 16 bins
    amount_bins = np.digitize(amounts, bins_amount) - 1
    amount_bins = np.clip(amount_bins, 0, 15)

    # Likelihoods from QCBM model
    likelihoods = np.array([qcbm_probs[b] for b in amount_bins])
    threshold = np.percentile(likelihoods, threshold_percentile)

    # Predict anomalies: likelihood < threshold
    predicted_labels = (likelihoods < threshold).astype(int)

    # Compute evaluation metrics
    precision = precision_score(labels, predicted_labels, zero_division=0)
    recall = recall_score(labels, predicted_labels, zero_division=0)
    f1 = f1_score(labels, predicted_labels, zero_division=0)
    roc_auc = roc_auc_score(labels, 1 - likelihoods)

    return precision, recall, f1, roc_auc, likelihoods

# Test multiple thresholds (e.g., 1%, 3%, 5%, 10%)
thresholds = [1, 3, 5, 10]
results = []

for thresh in thresholds:
    print(f"\n Evaluating QCBM Anomaly Detection at {thresh}% Threshold...")
    precision, recall, f1, roc_auc, likelihoods = evaluate_anomaly_detection(
        qcbm_probs_coarse, threshold_percentile=thresh, amounts=amounts_normalized, labels=labels
    )
    print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}, ROC-AUC: {roc_auc:.4f}")
    results.append((precision, recall, f1, roc_auc))

# Extract likelihoods at 5% threshold for visualization
_, _, _, _, likelihoods = evaluate_anomaly_detection(
    qcbm_probs_coarse, threshold_percentile=5, amounts=amounts_normalized, labels=labels
)

# Visualize likelihood distributions (Figure 4)
normal_likelihoods = likelihoods[labels == 0]
anomalous_likelihoods = likelihoods[labels == 1]

plt.figure(figsize=(12, 6))
plt.hist(normal_likelihoods, bins=30, alpha=0.5, label='Normal Transactions', color='blue', density=True)
plt.hist(anomalous_likelihoods, bins=30, alpha=0.5, label='Anomalous Transactions', color='red', density=True)
plt.xlabel('Likelihood')
plt.ylabel('Density')
plt.title('Figure 4: Likelihood Distribution — Normal vs. Anomalous Transactions')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('Figure_4_Likelihood_Distribution.png')
plt.close()

print(" Cell 6 completed: QCBM anomaly detection evaluated and Figure 4 saved.")



 Evaluating QCBM Anomaly Detection at 1% Threshold...
Precision: 0.0000, Recall: 0.0000, F1: 0.0000, ROC-AUC: 0.4833

 Evaluating QCBM Anomaly Detection at 3% Threshold...
Precision: 0.1071, Recall: 0.0600, F1: 0.0769, ROC-AUC: 0.4833

 Evaluating QCBM Anomaly Detection at 5% Threshold...
Precision: 0.0800, Recall: 0.0800, F1: 0.0800, ROC-AUC: 0.4833

 Evaluating QCBM Anomaly Detection at 10% Threshold...
Precision: 0.1122, Recall: 0.2200, F1: 0.1486, ROC-AUC: 0.4833
 Cell 6 completed: QCBM anomaly detection evaluated and Figure 4 saved.


In [7]:
# Cell 7: Classical Comparison with Isolation Forest (1D)
# Purpose: Train Isolation Forest and evaluate anomaly detection performance (1D)

from sklearn.ensemble import IsolationForest

# Reshape normalized amount for Isolation Forest
features = amounts_normalized.reshape(-1, 1)

# Train Isolation Forest only on normal transactions
iso_forest = IsolationForest(contamination=0.05, random_state=42)
iso_forest.fit(features[labels == 0])  # Only fit on normal transactions

# Predict anomaly scores for all data
iso_scores = iso_forest.decision_function(features)
iso_likelihoods = -iso_scores  # More negative = more anomalous

# Evaluate Isolation Forest at the same thresholds
iso_results = []

for thresh in thresholds:
    print(f"\n Evaluating Isolation Forest at {thresh}% Threshold...")
    threshold = np.percentile(iso_likelihoods, thresh)
    predicted_labels = (iso_likelihoods > threshold).astype(int)  # Higher score = more anomalous

    precision = precision_score(labels, predicted_labels, zero_division=0)
    recall = recall_score(labels, predicted_labels, zero_division=0)
    f1 = f1_score(labels, predicted_labels, zero_division=0)
    roc_auc = roc_auc_score(labels, iso_likelihoods)

    print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}, ROC-AUC: {roc_auc:.4f}")
    iso_results.append((precision, recall, f1, roc_auc))

print(" Cell 7 completed: Isolation Forest evaluation (1D) finished.")



 Evaluating Isolation Forest at 1% Threshold...
Precision: 0.0505, Recall: 1.0000, F1: 0.0962, ROC-AUC: 0.5457

 Evaluating Isolation Forest at 3% Threshold...
Precision: 0.0414, Recall: 0.7600, F1: 0.0785, ROC-AUC: 0.5457

 Evaluating Isolation Forest at 5% Threshold...
Precision: 0.0414, Recall: 0.7600, F1: 0.0785, ROC-AUC: 0.5457

 Evaluating Isolation Forest at 10% Threshold...
Precision: 0.0422, Recall: 0.7600, F1: 0.0800, ROC-AUC: 0.5457
 Cell 7 completed: Isolation Forest evaluation (1D) finished.


In [8]:
# Extended Cell 7: Plot Isolation Forest Performance vs. Threshold
# Purpose: Generate Figure 5 to visualize metrics across thresholds

# Unpack metrics from iso_results
precisions, recalls, f1_scores, roc_aucs = zip(*iso_results)

# Plotting
plt.figure(figsize=(14, 6))
x_vals = thresholds

plt.plot(x_vals, precisions, marker='o', label='Precision', linestyle='-')
plt.plot(x_vals, recalls, marker='s', label='Recall', linestyle='--')
plt.plot(x_vals, f1_scores, marker='^', label='F1-Score', linestyle='-.')
plt.plot(x_vals, roc_aucs, marker='d', label='ROC-AUC', linestyle=':')

plt.xlabel('Anomaly Threshold Percentile (%)')
plt.ylabel('Score')
plt.title('Figure 5: Isolation Forest Metrics vs. Threshold')
plt.xticks(x_vals)
plt.ylim(0, 1.05)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig('Figure_5_IsolationForest_Metrics_vs_Threshold.png')
plt.close()

print(" Figure 5 saved: Isolation Forest performance metrics across thresholds.")


 Figure 5 saved: Isolation Forest performance metrics across thresholds.


In [9]:
# Cell 8: Adaptive Thresholding for QCBM (1D)
# Purpose: Maximize F1-score by finding the optimal threshold on QCBM likelihoods

from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

# Reuse binning method to compute likelihoods again
bins_amount = np.linspace(0, 1, 17)
amount_bins = np.digitize(amounts_normalized, bins_amount) - 1
amount_bins = np.clip(amount_bins, 0, 15)

# Likelihoods from QCBM coarse probabilities
likelihoods = np.array([qcbm_probs_coarse[idx] for idx in amount_bins])

# Range of thresholds to search
thresholds_adaptive = np.linspace(np.min(likelihoods), np.max(likelihoods), 200)

# Track metrics for plotting or evaluation
f1_scores = []
precision_scores = []
recall_scores = []

best_f1 = 0
best_threshold = 0
best_metrics = (0, 0, 0)

# Sweep over thresholds to find max F1
for t in thresholds_adaptive:
    preds = (likelihoods < t).astype(int)
    precision = precision_score(labels, preds, zero_division=0)
    recall = recall_score(labels, preds, zero_division=0)
    f1 = f1_score(labels, preds, zero_division=0)

    f1_scores.append(f1)
    precision_scores.append(precision)
    recall_scores.append(recall)

    if f1 > best_f1:
        best_f1 = f1
        best_threshold = t
        best_metrics = (precision, recall, f1)

# Compute ROC-AUC at best threshold
final_preds = (likelihoods < best_threshold).astype(int)
roc_auc_final = roc_auc_score(labels, 1 - likelihoods)

# Print optimal threshold and scores
print(f"\n Optimal QCBM Threshold (Likelihood < {best_threshold:.6f}):")
print(f"Precision: {best_metrics[0]:.4f}, Recall: {best_metrics[1]:.4f}, F1-Score: {best_metrics[2]:.4f}, ROC-AUC: {roc_auc_final:.4f}")



 Optimal QCBM Threshold (Likelihood < 0.046254):
Precision: 0.1042, Recall: 0.3000, F1-Score: 0.1546, ROC-AUC: 0.4833


In [10]:
# Extended Cell 8: F1-score vs. Threshold Plot
# Purpose: Generate Figure 6 to visualize adaptive thresholding effectiveness for QCBM (1D)

plt.figure(figsize=(12, 6))
plt.plot(thresholds_adaptive, f1_scores, label='F1-Score', color='purple')
plt.xlabel('Likelihood Threshold')
plt.ylabel('F1-Score')
plt.title('Figure 6: F1-Score vs. QCBM Threshold (1D)')
plt.grid(True, alpha=0.3)
plt.axvline(best_threshold, color='red', linestyle='--', label=f'Best Threshold = {best_threshold:.4f}')
plt.legend()
plt.tight_layout()
plt.savefig('Figure_6_F1_vs_QCBM_Threshold.png')
plt.close()

print(" Figure 6 saved: F1-score vs. threshold for QCBM (1D).")


 Figure 6 saved: F1-score vs. threshold for QCBM (1D).


In [11]:
# Cell 9: Data Preprocessing with PCA Compression
# Purpose: Apply PCA to reduce feature dimensionality and define 2D target distribution (16x16)

from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
import numpy as np

# Extract the 'Time' feature
times = data['Time'].values
times_norm = MinMaxScaler().fit_transform(times.reshape(-1, 1)).flatten()

# Select features to use for PCA
features_to_use = ['Amount', 'Time'] + [f'V{i}' for i in range(1, 28)]
data_subset = data[features_to_use]

# Normalize all selected features
scaler = MinMaxScaler()
normalized_features = scaler.fit_transform(data_subset)

# Apply PCA to reduce features to 2D
pca = PCA(n_components=2)
pca_result = pca.fit_transform(normalized_features)

# Extract the two PCA components
pca_x = pca_result[:, 0]
pca_y = pca_result[:, 1]

# Normalize the PCA components to [0, 1] for binning
pca_x_norm = MinMaxScaler().fit_transform(pca_x.reshape(-1, 1)).flatten()
pca_y_norm = MinMaxScaler().fit_transform(pca_y.reshape(-1, 1)).flatten()

# Bin data into 16x16 grid
bins_1d = np.linspace(0, 1, 17)
x_bins = np.digitize(pca_x_norm, bins_1d) - 1
y_bins = np.digitize(pca_y_norm, bins_1d) - 1

# Clip to stay within grid
x_bins = np.clip(x_bins, 0, 15)
y_bins = np.clip(y_bins, 0, 15)

# Compute target 2D distribution from normal transactions only
normal_indices = np.where(labels == 0)[0]
target_dist_2d, _, _ = np.histogram2d(
    x_bins[normal_indices],
    y_bins[normal_indices],
    bins=(16, 16),
    range=[[0, 16], [0, 16]],
    density=True
)

# Normalize
target_dist_2d = target_dist_2d / np.sum(target_dist_2d)

print(" Cell 9 completed: PCA applied and 2D target distribution defined.")


 Cell 9 completed: PCA applied and 2D target distribution defined.


In [12]:
# Cell 10: QCBM Circuit Definition for Multi-Feature (2D)
# Purpose: Reuse the same 8-qubit, 8-layer circuit for learning the 16x16 (256-bin) 2D distribution

# The same circuit from Cell 2 is reused here.
# 8 qubits can represent 256 states, sufficient for a 16x16 grid.

# No need to rebuild the circuit.
# Just a confirmation message.

print(" Cell 10 completed: Reusing the QCBM circuit from Cell 2 for 2D modeling.")


 Cell 10 completed: Reusing the QCBM circuit from Cell 2 for 2D modeling.


In [13]:
# Cell 11: Training Setup for Multi-Feature QCBM (2D)
# Purpose: Define cost function for 2D QCBM distribution matching

# We reuse jsd() and kl_div() functions from Cell 3

# Define cost function for 2D target and learned distributions
def cost_function_2d(params):
    # Assign parameters to the circuit
    param_dict = {theta[i]: params[i] for i in range(len(params))}
    qc_bound = qc.assign_parameters(param_dict)

    # Run simulation
    result = simulator.run(qc_bound, shots=50000).result()
    counts = result.get_counts()

    # Full probability vector (length 256)
    qcbm_probs_full = np.array([counts.get(f"{i:08b}", 0) / 50000 for i in range(256)])
    qcbm_probs_full += 1e-9  # Avoid zeros
    qcbm_probs_full /= np.sum(qcbm_probs_full)

    # Reshape to 16x16 for 2D grid
    qcbm_probs_2d = qcbm_probs_full.reshape(16, 16)
    qcbm_probs_2d /= np.sum(qcbm_probs_2d)

    # Flatten to compute divergence
    jsd_value = jsd(target_dist_2d.flatten(), qcbm_probs_2d.flatten())
    kl_value = kl_div(target_dist_2d.flatten(), qcbm_probs_2d.flatten())

    # Return total cost
    return jsd_value + kl_value


In [14]:
# Cell 12: Train Multi-Feature QCBM (2D)
# Purpose: Optimize QCBM parameters for 2D target distribution and generate Figure 7

import numpy as np
import matplotlib.pyplot as plt

# Training settings
maxiter = 250
learning_rate = 0.01
perturbation = 0.1

# Initial parameters
initial_params_2d = np.random.uniform(0, 2 * np.pi, len(theta))
params_2d = initial_params_2d.copy()

# Store cost values over iterations
cost_values_2d = []

# Custom SPSA-like loop (simple gradient approximation)
for i in range(maxiter):
    if i % 25 == 0:
        print(f"Iteration {i}/{maxiter}")

    # Compute cost at current params
    cost = cost_function_2d(params_2d)
    cost_values_2d.append(cost)

    # Generate symmetric perturbation
    delta = np.random.choice([-1, 1], size=len(params_2d)) * perturbation
    params_plus = params_2d + delta
    params_minus = params_2d - delta

    cost_plus = cost_function_2d(params_plus)
    cost_minus = cost_function_2d(params_minus)

    # Gradient estimate and parameter update
    gradient = (cost_plus - cost_minus) / (2 * delta)
    params_2d = params_2d - learning_rate * gradient

# Final QCBM with trained parameters
param_dict_2d = {theta[i]: params_2d[i] for i in range(len(params_2d))}
qc_optimized_2d = qc.assign_parameters(param_dict_2d)

# Simulate final trained circuit
result_sim_2d = simulator.run(qc_optimized_2d, shots=50000).result()
counts_2d = result_sim_2d.get_counts()

# Convert results into 2D probability grid
qcbm_probs_2d_full = np.array([counts_2d.get(f"{i:08b}", 0) / 50000 for i in range(256)])
qcbm_probs_2d_full += 1e-9
qcbm_probs_2d_full /= np.sum(qcbm_probs_2d_full)
qcbm_probs_2d = qcbm_probs_2d_full.reshape(16, 16)
qcbm_probs_2d /= np.sum(qcbm_probs_2d)

# Final evaluation
jsd_final_2d = jsd(target_dist_2d.flatten(), qcbm_probs_2d.flatten())
kl_final_2d = kl_div(target_dist_2d.flatten(), qcbm_probs_2d.flatten())

print(" Cell 12 completed: 2D QCBM training finished.")
print(f"Final JSD (2D): {jsd_final_2d:.4f}, Final KL (2D): {kl_final_2d:.4f}, Total Cost: {jsd_final_2d + kl_final_2d:.4f}")

# Plot cost function (Figure 7)
plt.figure(figsize=(12, 6))
plt.plot(range(len(cost_values_2d)), cost_values_2d, label='Cost (JSD + KL)', color='blue')
plt.xlabel('Iteration')
plt.ylabel('Cost')
plt.title('Figure 7: Cost Function Over Iterations (2D QCBM Training)')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig('Figure_7_Cost_Function_2D.png')
plt.close()


Iteration 0/250
Iteration 25/250
Iteration 50/250
Iteration 75/250
Iteration 100/250
Iteration 125/250
Iteration 150/250
Iteration 175/250
Iteration 200/250
Iteration 225/250
 Cell 12 completed: 2D QCBM training finished.
Final JSD (2D): 0.4454, Final KL (2D): 1.5806, Total Cost: 2.0259


In [15]:
# Cell 13: Visualize 2D Distributions (Target vs. QCBM)
# Purpose: Compare target vs. learned 2D distributions using heatmaps (Figure 8)

import seaborn as sns
import matplotlib.pyplot as plt

# Create a side-by-side heatmap comparison
plt.figure(figsize=(16, 7))

# Target distribution heatmap
plt.subplot(1, 2, 1)
sns.heatmap(target_dist_2d, cmap="Blues", cbar=True, square=True)
plt.title("Target 2D Distribution (PCA Components)")
plt.xlabel("Component 2 Bin")
plt.ylabel("Component 1 Bin")

# QCBM-learned distribution heatmap
plt.subplot(1, 2, 2)
sns.heatmap(qcbm_probs_2d, cmap="Greens", cbar=True, square=True)
plt.title("QCBM 2D Distribution (PCA Components)")
plt.xlabel("Component 2 Bin")
plt.ylabel("Component 1 Bin")

# Save the heatmap comparison
plt.tight_layout()
plt.savefig('Figure_8_2D_Distribution_Comparison.png')
plt.close()

print(" Cell 13 completed: Figure 8 generated — QCBM vs. Target 2D heatmaps.")


 Cell 13 completed: Figure 8 generated — QCBM vs. Target 2D heatmaps.


In [16]:
# Cell 14: Anomaly Detection with Multi-Feature QCBM (2D)
# Purpose: Evaluate anomaly detection using 2D QCBM and visualize likelihood distributions (Figure 9)

from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
import numpy as np
import matplotlib.pyplot as plt

# Compute likelihoods for each transaction based on 2D bin
likelihoods_2d = np.array([qcbm_probs_2d[x_bins[i], y_bins[i]] for i in range(len(labels))])

# Evaluate performance at multiple anomaly thresholds
thresholds = [1, 3, 5, 7, 10]
qcbm_2d_results = []

for thresh in thresholds:
    print(f"\n Evaluating Multi-Feature QCBM at {thresh}% Threshold...")
    threshold = np.percentile(likelihoods_2d, thresh)
    predicted_labels = (likelihoods_2d < threshold).astype(int)

    precision = precision_score(labels, predicted_labels, zero_division=0)
    recall = recall_score(labels, predicted_labels, zero_division=0)
    f1 = f1_score(labels, predicted_labels, zero_division=0)
    roc_auc = roc_auc_score(labels, 1 - likelihoods_2d)

    print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}, ROC-AUC: {roc_auc:.4f}")
    qcbm_2d_results.append((precision, recall, f1, roc_auc))

# Separate likelihoods for visualization
normal_likelihoods_2d = likelihoods_2d[labels == 0]
anomalous_likelihoods_2d = likelihoods_2d[labels == 1]

# Plot distributions (Figure 9)
plt.figure(figsize=(12, 6))
plt.hist(normal_likelihoods_2d, bins=30, alpha=0.6, label='Normal Transactions', color='blue', density=True)
plt.hist(anomalous_likelihoods_2d, bins=30, alpha=0.6, label='Anomalous Transactions', color='red', density=True)
plt.xlabel('Joint Likelihood (PCA Bins)')
plt.ylabel('Density')
plt.title('Figure 9: Likelihood Distribution — QCBM (2D)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('Figure_9_QCBM_2D_Likelihood_Distribution.png')
plt.close()

print(" Cell 14 completed: 2D anomaly detection performed and Figure 9 saved.")



 Evaluating Multi-Feature QCBM at 1% Threshold...
Precision: 1.0000, Recall: 0.2000, F1: 0.3333, ROC-AUC: 0.9135

 Evaluating Multi-Feature QCBM at 3% Threshold...
Precision: 0.8667, Recall: 0.5200, F1: 0.6500, ROC-AUC: 0.9135

 Evaluating Multi-Feature QCBM at 5% Threshold...
Precision: 0.7907, Recall: 0.6800, F1: 0.7312, ROC-AUC: 0.9135

 Evaluating Multi-Feature QCBM at 7% Threshold...
Precision: 0.5373, Recall: 0.7200, F1: 0.6154, ROC-AUC: 0.9135

 Evaluating Multi-Feature QCBM at 10% Threshold...
Precision: 0.4105, Recall: 0.7800, F1: 0.5379, ROC-AUC: 0.9135
 Cell 14 completed: 2D anomaly detection performed and Figure 9 saved.


In [17]:
# Cell 15: Classical Model Comparison Using Multi-Features (2D)
# Purpose: Train and evaluate Isolation Forest on PCA 2D features

from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

# Combine PCA components into a 2D feature array
features_2d = np.column_stack((pca_x_norm, pca_y_norm))

# Train Isolation Forest on normal transactions only
iso_forest_2d = IsolationForest(contamination=0.05, random_state=42)
iso_forest_2d.fit(features_2d[labels == 0])

# Compute anomaly scores and likelihoods
iso_scores_2d = iso_forest_2d.decision_function(features_2d)
iso_likelihoods_2d = -iso_scores_2d  # More positive = more anomalous

# Predict labels for all transactions
iso_predictions_2d = iso_forest_2d.predict(features_2d)
iso_predictions_2d = np.where(iso_predictions_2d == -1, 1, 0)  # 1 = anomaly

# Evaluate across thresholds (1%, 3%, 5%, 7%, 10%)
iso_2d_results = []

for thresh in thresholds:
    print(f"\n Evaluating Isolation Forest (2D) at {thresh}% Threshold...")
    threshold = np.percentile(iso_likelihoods_2d, thresh)
    predicted_labels = (iso_likelihoods_2d > threshold).astype(int)

    precision = precision_score(labels, predicted_labels, zero_division=0)
    recall = recall_score(labels, predicted_labels, zero_division=0)
    f1 = f1_score(labels, predicted_labels, zero_division=0)
    roc_auc = roc_auc_score(labels, iso_likelihoods_2d)

    print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}, ROC-AUC: {roc_auc:.4f}")
    iso_2d_results.append((precision, recall, f1, roc_auc))

print(" Cell 15 completed: Isolation Forest (2D) evaluation finished.")



 Evaluating Isolation Forest (2D) at 1% Threshold...
Precision: 0.0505, Recall: 1.0000, F1: 0.0962, ROC-AUC: 0.9292

 Evaluating Isolation Forest (2D) at 3% Threshold...
Precision: 0.0515, Recall: 1.0000, F1: 0.0980, ROC-AUC: 0.9292

 Evaluating Isolation Forest (2D) at 5% Threshold...
Precision: 0.0526, Recall: 1.0000, F1: 0.1000, ROC-AUC: 0.9292

 Evaluating Isolation Forest (2D) at 7% Threshold...
Precision: 0.0538, Recall: 1.0000, F1: 0.1020, ROC-AUC: 0.9292

 Evaluating Isolation Forest (2D) at 10% Threshold...
Precision: 0.0544, Recall: 0.9800, F1: 0.1032, ROC-AUC: 0.9292
 Cell 15 completed: Isolation Forest (2D) evaluation finished.


In [18]:
# Cell 16: Adaptive Thresholding for QCBM 2D
# Purpose: Find optimal threshold (based on F1-score) using 2D QCBM likelihoods and generate Figure 10

# Prepare to scan over possible thresholds
thresholds_adaptive = np.linspace(np.min(likelihoods_2d), np.max(likelihoods_2d), 200)
f1_scores = []
precision_scores = []
recall_scores = []

# Tracking best metrics
best_f1 = 0
best_threshold = 0
best_metrics = (0, 0, 0)

# Sweep over thresholds to optimize F1-score
for t in thresholds_adaptive:
    preds = (likelihoods_2d < t).astype(int)
    precision = precision_score(labels, preds, zero_division=0)
    recall = recall_score(labels, preds, zero_division=0)
    f1 = f1_score(labels, preds, zero_division=0)

    f1_scores.append(f1)
    precision_scores.append(precision)
    recall_scores.append(recall)

    if f1 > best_f1:
        best_f1 = f1
        best_threshold = t
        best_metrics = (precision, recall, f1)

# Compute ROC-AUC using optimal threshold
final_preds = (likelihoods_2d < best_threshold).astype(int)
roc_auc_final = roc_auc_score(labels, 1 - likelihoods_2d)

# Print final optimal threshold and metrics
print(f"\n Optimal Threshold (2D QCBM Likelihood < {best_threshold:.6f}):")
print(f"Precision: {best_metrics[0]:.4f}, Recall: {best_metrics[1]:.4f}, F1-Score: {best_metrics[2]:.4f}, ROC-AUC: {roc_auc_final:.4f}")

# Plot F1-score vs. threshold (Figure 10)
plt.figure(figsize=(12, 6))
plt.plot(thresholds_adaptive, f1_scores, label='F1-Score', color='purple')
plt.xlabel('QCBM 2D Likelihood Threshold')
plt.ylabel('F1-Score')
plt.title('Figure 10: Adaptive Thresholding for QCBM 2D')
plt.grid(True, alpha=0.3)
plt.axvline(best_threshold, color='red', linestyle='--', label=f'Best Threshold = {best_threshold:.4f}')
plt.legend()
plt.tight_layout()
plt.savefig('Figure_10_Adaptive_Thresholding_QCBM_2D.png')
plt.close()

print(" Cell 16 completed: Optimal threshold selected and Figure 10 saved.")



 Optimal Threshold (2D QCBM Likelihood < 0.003160):
Precision: 0.8108, Recall: 0.6000, F1-Score: 0.6897, ROC-AUC: 0.9135
 Cell 16 completed: Optimal threshold selected and Figure 10 saved.


In [19]:
# Cell 17: Anomaly Location Heatmap
# Purpose: Show fraud (anomaly) locations in 2D PCA bin grid, compared to QCBM learned distribution (Figure 11)

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Count anomaly frequencies per 2D bin
anomaly_positions = [(x_bins[i], y_bins[i]) for i in range(len(labels)) if labels[i] == 1]

anomaly_heatmap = np.zeros((16, 16))
for x, y in anomaly_positions:
    anomaly_heatmap[x, y] += 1

# Normalize for visualization
anomaly_heatmap /= (np.sum(anomaly_heatmap) + 1e-10)

# Plot side-by-side: QCBM learned 2D distribution vs. anomaly locations
plt.figure(figsize=(16, 7))

# QCBM-learned 2D distribution
plt.subplot(1, 2, 1)
sns.heatmap(qcbm_probs_2d, cmap="Greens", cbar=True, square=True)
plt.title("QCBM Learned Distribution (PCA Space)")
plt.xlabel("Component 2 Bin")
plt.ylabel("Component 1 Bin")

# Fraud location map (from labeled anomalies)
plt.subplot(1, 2, 2)
sns.heatmap(anomaly_heatmap, cmap="Reds", cbar=True, square=True)
plt.title("Anomalous Transaction Locations")
plt.xlabel("Component 2 Bin")
plt.ylabel("Component 1 Bin")

plt.tight_layout()
plt.savefig('Figure_11_Anomaly_Location_Heatmap.png')
plt.close()

print(" Cell 17 completed: Figure 11 saved — QCBM vs. anomaly distribution heatmaps.")


 Cell 17 completed: Figure 11 saved — QCBM vs. anomaly distribution heatmaps.


In [20]:
# Cell 18: QCBM vs. Target and Anomalies – Zoomed to First 50 Bins
# Purpose: Compare QCBM distribution to target and anomalies for top 50 bins (Figures 12 and 13)

import numpy as np
import matplotlib.pyplot as plt

# Flatten both 2D distributions
qcbm_flat = qcbm_probs_2d.flatten()
target_flat = target_dist_2d.flatten()

# Select top 50 bins for clarity
x = np.arange(50)
qcbm_top50 = qcbm_flat[:50]
target_top50 = target_flat[:50]

# --- Figure 12: QCBM vs. Target Distribution ---
plt.figure(figsize=(14, 6))
bar_width = 0.4
plt.bar(x - bar_width/2, target_top50, bar_width, label='Target Distribution', alpha=0.7, color='blue')
plt.bar(x + bar_width/2, qcbm_top50, bar_width, label='QCBM Distribution', alpha=0.7, color='green')
plt.xlabel('Flattened 2D Bin Index (0–49)')
plt.ylabel('Probability')
plt.title('Figure 12: Target vs. QCBM Distribution (First 50 Bins)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('Figure_12_QCBM_vs_Target_Top50.png')
plt.close()

# --- Figure 13: QCBM vs. Anomaly Frequency ---
# Count anomalies in each bin
anomaly_counts = np.zeros((16, 16))
for i in range(len(labels)):
    if labels[i] == 1:
        x_bin, y_bin = x_bins[i], y_bins[i]
        anomaly_counts[x_bin, y_bin] += 1

# Flatten and normalize for top 50 visualization
anomaly_flat = anomaly_counts.flatten()
anomaly_freq_top50 = anomaly_flat[:50] / (np.sum(anomaly_flat) + 1e-10)

plt.figure(figsize=(14, 6))
plt.bar(x - bar_width/2, qcbm_top50, bar_width, label='QCBM Learned Probability', alpha=0.7, color='green')
plt.bar(x + bar_width/2, anomaly_freq_top50, bar_width, label='Anomalous Frequency', alpha=0.7, color='red')
plt.xlabel('Flattened 2D Bin Index (0–49)')
plt.ylabel('Probability / Frequency')
plt.title('Figure 13: QCBM Probability vs. Anomaly Frequency (First 50 Bins)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('Figure_13_QCBM_vs_Anomaly_Top50.png')
plt.close()

print(" Cell 18 completed: Figures 12 and 13 generated — detailed QCBM vs. Target/Anomaly comparison.")


 Cell 18 completed: Figures 12 and 13 generated — detailed QCBM vs. Target/Anomaly comparison.


In [21]:
# Cell 19: Metrics Summary Using Stored Results
# Purpose: Create comparison table using stored evaluation results

import pandas as pd

# Use 5% threshold (3rd index = 2)
qcbm_1d_metrics = results[2]           # from Cell 6
iso_1d_metrics = iso_results[2]        # from Cell 7
qcbm_2d_metrics = best_metrics         # from Cell 16 (optimal threshold)
iso_2d_metrics = iso_2d_results[2]     # from Cell 15

# Compile DataFrame
df_metrics = pd.DataFrame({
    'Model': [
        'QCBM (1D)',
        'Isolation Forest (1D)',
        'QCBM (2D)',
        'Isolation Forest (2D)'
    ],
    'Precision': [
        qcbm_1d_metrics[0],
        iso_1d_metrics[0],
        qcbm_2d_metrics[0],
        iso_2d_metrics[0]
    ],
    'Recall': [
        qcbm_1d_metrics[1],
        iso_1d_metrics[1],
        qcbm_2d_metrics[1],
        iso_2d_metrics[1]
    ],
    'F1-Score': [
        qcbm_1d_metrics[2],
        iso_1d_metrics[2],
        qcbm_2d_metrics[2],
        iso_2d_metrics[2]
    ],
    'ROC-AUC': [
        qcbm_1d_metrics[3],
        iso_1d_metrics[3],
        roc_auc_final,             # from Cell 16
        iso_2d_metrics[3]
    ]
})

# Display table
print("\n Anomaly Detection Performance Summary:\n")
print(df_metrics.to_string(index=False, float_format="%.4f"))



 Anomaly Detection Performance Summary:

                Model  Precision  Recall  F1-Score  ROC-AUC
            QCBM (1D)     0.0800  0.0800    0.0800   0.4833
Isolation Forest (1D)     0.0414  0.7600    0.0785   0.5457
            QCBM (2D)     0.8108  0.6000    0.6897   0.9135
Isolation Forest (2D)     0.0526  1.0000    0.1000   0.9292


In [22]:
# Extended Cell 19: Generate Figure 14 — Model Comparison via Bar Plots

import matplotlib.pyplot as plt
import numpy as np

# Extract data
models = df_metrics['Model'].values
metrics = ['Precision', 'Recall', 'F1-Score', 'ROC-AUC']

# Create grouped bar plot
x = np.arange(len(models))  # label locations
bar_width = 0.1

# Plot setup
plt.figure(figsize=(14, 6))

# Plot each metric as a group
for i, metric in enumerate(metrics):
    plt.bar(x + i * bar_width, df_metrics[metric], width=bar_width, label=metric)

# Formatting
plt.xlabel('Model')
plt.ylabel('Score')
plt.title('Figure 14: Anomaly Detection Model Comparison (QCBM vs Isolation Forest)')
plt.xticks(x + bar_width * 1.5, models)
plt.ylim(0, 1.05)
plt.legend()
plt.grid(True, axis='y', alpha=0.3)
plt.tight_layout()

# Save figure
plt.savefig('Figure_14_Model_Comparison_BarPlot.png')
plt.close()

print(" Figure 14 saved: Model comparison across Precision, Recall, F1, and ROC-AUC.")


 Figure 14 saved: Model comparison across Precision, Recall, F1, and ROC-AUC.
